In [0]:
%pip install httpx-sse httpx
dbutils.library.restartPython()

In [0]:
%run ./config

In [0]:
SECRET_SCOPE = get_widget_param("secret_scope", "eventhub")
SECRET_KEY = get_widget_param("secret_key", "eh-connection-string")
EH_NAME = get_widget_param("eh_name", "roksolana-wikipedia-recentchange")
WIKI_FILTER = get_widget_param("wiki_filter", "en.wikipedia.org")
MAX_EVENTS = int(get_widget_param("max_events", "2000"))
RETRY_TOTAL = int(get_widget_param("retry_total", "3"))
RETRY_BACKOFF_FACTOR = int(get_widget_param("retry_backoff_factor", "2"))
MAX_RECONNECT_ATTEMPTS = int(get_widget_param("max_reconnect_attempts", "5"))

EH_CONN_STR = get_eventhub_connection_string(SECRET_SCOPE, SECRET_KEY)

In [0]:
import json
import asyncio
import logging
import httpx
from httpx_sse import aconnect_sse
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData

logger = logging.getLogger("wikipedia_eventhub_producer")
logger.setLevel(logging.INFO)

FIELDS = ["id", "type", "title", "user", "bot", "minor", "timestamp", "wiki", "server_name", "length"]
RECONNECTABLE_ERRORS = (httpx.ReadError, httpx.ConnectError, httpx.RemoteProtocolError, httpx.ReadTimeout)


async def stream_events(producer, state):
    headers = {"User-Agent": "DatabricksLab3Producer/1.0 (roksolana.shendiu770@softserve.academy)"}

    async with httpx.AsyncClient(headers=headers, timeout=httpx.Timeout(connect=10, read=60, write=10, pool=10)) as http_client:
        async with aconnect_sse(http_client, "GET", "https://stream.wikimedia.org/v2/stream/recentchange") as event_source:
            async for sse in event_source.aiter_sse():
                if not sse.data:
                    continue

                try:
                    record = json.loads(sse.data)
                except json.JSONDecodeError:
                    logger.warning("Skipped malformed JSON event")
                    continue

                if record.get("server_name") != WIKI_FILTER:
                    state["filtered_count"] += 1
                    continue

                payload = {k: record.get(k) for k in FIELDS}

                try:
                    state["batch"].add(EventData(json.dumps(payload)))
                except ValueError:
                    task = asyncio.create_task(producer.send_batch(state["batch"]))
                    state["pending_sends"].append(task)
                    state["batch"] = await producer.create_batch()
                    state["batch"].add(EventData(json.dumps(payload)))

                state["sent_count"] += 1
                if state["sent_count"] >= MAX_EVENTS:
                    return


async def run():
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EH_CONN_STR,
        eventhub_name=EH_NAME,
        retry_total=RETRY_TOTAL,
        retry_backoff_factor=RETRY_BACKOFF_FACTOR,
    )

    state = {"sent_count": 0, "filtered_count": 0, "batch": None, "pending_sends": []}

    logger.info("Starting Wikipedia stream producer, target=%s, max_events=%d", EH_NAME, MAX_EVENTS)

    async with producer:
        state["batch"] = await producer.create_batch()
        attempt = 0

        while state["sent_count"] < MAX_EVENTS and attempt < MAX_RECONNECT_ATTEMPTS:
            try:
                await stream_events(producer, state)
                break
            except RECONNECTABLE_ERRORS as e:
                attempt += 1
                logger.warning(
                    "SSE connection lost at %d/%d events sent (reconnect attempt %d/%d): %s",
                    state["sent_count"], MAX_EVENTS, attempt, MAX_RECONNECT_ATTEMPTS, e
                )
                if attempt < MAX_RECONNECT_ATTEMPTS:
                    await asyncio.sleep(min(2 ** attempt, 30))

        if attempt >= MAX_RECONNECT_ATTEMPTS and state["sent_count"] < MAX_EVENTS:
            logger.error("Max reconnect attempts reached. Sent %d/%d events.", state["sent_count"], MAX_EVENTS)

        if len(state["batch"]) > 0:
            await producer.send_batch(state["batch"])

        if state["pending_sends"]:
            results = await asyncio.gather(*state["pending_sends"], return_exceptions=True)
            failed = [r for r in results if isinstance(r, Exception)]
            if failed:
                logger.error("Failed to send %d background batch(es): %s", len(failed), failed)
            logger.info("Awaited %d background batch send(s), %d failed.", len(results), len(failed))

    logger.info(
        "Producer finished. sent=%d, filtered_out=%d, total_events_seen=%d",
        state["sent_count"], state["filtered_count"],
        state["sent_count"] + state["filtered_count"]
    )


await run()